In [4]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence 
import numpy as np
import random
import re
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

In [5]:
def seed_everything(seed = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything()

df = pd.read_csv('train.tsv', sep='\t').drop(columns = 'eebbqej')
df.columns = ['text', 'labels']

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    return text.strip()

df['text'] = df['text'].apply(lambda x:clean_text(x))
texts = df['text'].apply(str.split).tolist()

def parse_labels(lbl):
    if isinstance(lbl,int):
        return [lbl]
    return list(map(int,str(lbl).split(',')))

labels = [parse_labels(l) for l in df['labels']]
mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(labels)

class ECdataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def build_vocab_and_embeddings(texts, glove_path='glove.6B.100d.txt', embed_dim = 100):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    idx = 2
    for sentence in texts:
        for word in sentence:
            if word not in vocab:
                vocab[word] = idx
                idx += 1

    print("Loading GloVe vectors ...")
    glove = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            glove[parts[0]] = np.array(parts[1:], dtype=np.float32)

    vocab_size = len(vocab)
    embedding_matrix = np.zeros((vocab_size, embed_dim)) 

    for word, i in vocab.items():
        if word in glove:
            embedding_matrix[i] = glove[word]
        else:
            embedding_matrix[i] = np.random.normal(scale=0.6, size=(embed_dim,))

    return vocab, torch.tensor(embedding_matrix, dtype=torch.float32)

vocab, embedding_matrix = build_vocab_and_embeddings(texts)

def encode_texts(texts, vocab):
    encoded = []
    for sentences in texts:
        encoded.append(torch.tensor([vocab.get(word,vocab["<UNK>"]) for word in sentences], dtype=torch.long))

    return pad_sequence(encoded, batch_first=True, padding_value=vocab["<PAD>"])

X = encode_texts(texts, vocab)
y = torch.tensor(labels, dtype=torch.float32)

Loading GloVe vectors ...


In [6]:
X_train, X_val , y_train , y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y.sum(dim=1)>1)
train_data = ECdataset(X_train, y_train)
val_data = ECdataset(X_val, y_val)

train_loader = DataLoader(train_data,batch_size = 32, shuffle = True)
val_loader = DataLoader(val_data,batch_size = 32, shuffle = True)

In [7]:
print(train_data.__len__())

34727


In [8]:
class EmotionClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, output_dim):
        super().__init__()
        _, embedding_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)
        self.LSTM = nn.LSTM(embedding_dim, hidden_dim,num_layers=1, batch_first= True, bidirectional = True)
        self.attn = nn.MultiheadAttention(hidden_dim*2, 4, batch_first=True)
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(hidden_dim*2, output_dim)

        
    def forward(self, x):
        embedded = self.embedding(x)
        out, (h,c) = self.LSTM(embedded)
        max_pool = torch.max(out, dim=1)[0]
        out,_ = self.attn(out,out,out)
        out = self.dropout(out)
        return self.fc(max_pool)

In [9]:
device = 'cuda' if torch.cuda.is_available else 'cpu'

In [10]:
model = EmotionClassifier(embedding_matrix, hidden_dim=128, output_dim=labels.shape[1]).to(device)

In [11]:
print(model)

EmotionClassifier(
  (embedding): Embedding(27323, 100)
  (LSTM): LSTM(100, 128, batch_first=True, bidirectional=True)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=28, bias=True)
)


In [58]:

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model1.parameters(), lr=0.0001)

In [59]:
def compute_subset_accuracy(y_true, y_pred_logits, threshold=0.5):
    with torch.no_grad():
        y_pred = (torch.sigmoid(y_pred_logits) > threshold).float()
        
        # Compare whole rows (samples)
        correct_per_sample = (y_pred == y_true).all(dim=1)
        
        # Subset accuracy = % of samples where ALL labels are correct
        subset_accuracy = correct_per_sample.float().mean().item()
        
        return subset_accuracy


In [60]:
best_val_f1 = 0
patience = 5
patience_counter = 0
best_model_path = 'best_emotion_model.pt'

history = {
    'train_loss':[],
    'val_loss': [],
    'train_acc': [],
    'val_acc': []
}

In [62]:
for epoch in range(50):
    # Training phase
    model1.train()
    total_loss = 0
    all_train_preds = []
    all_train_labels = []
    
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/30 [Train]")
    for xb, yb in train_loop:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        preds = model1(xb)
        loss = criterion(preds, yb)
        loss.backward()

        optimizer.step()
        total_loss += loss.item()
        
        all_train_preds.append(preds.detach())
        all_train_labels.append(yb)
        
        train_loop.set_postfix(loss=f"{loss.item():.4f}")

    all_train_preds = torch.cat(all_train_preds)
    all_train_labels = torch.cat(all_train_labels)

    # Validation phase
    model1.eval()
    val_loss = 0

    all_val_preds = []
    all_val_labels = []
    
    with torch.no_grad():
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/30 [Valid]")
        for xb, yb in val_loop:
            xb = xb.to(device)
            yb = yb.to(device)
            
            preds = model1(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item()
            
            all_val_preds.append(preds)
            all_val_labels.append(yb)
            
            val_loop.set_postfix(loss=f"{loss.item():.4f}")
    
    all_val_preds = torch.cat(all_val_preds)
    all_val_labels = torch.cat(all_val_labels)

    train_acc = compute_subset_accuracy(all_train_labels, all_train_preds)
    val_acc = compute_subset_accuracy(all_val_labels, all_val_preds)

    train_loss = total_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")
    print(f"            Train Acc = {train_acc*100:.4f}, Val Acc = {val_acc*100:.4f}")

    # Early stopping logic
    val_preds_binary = (torch.sigmoid(all_val_preds) > 0.5).float().cpu().numpy()
    val_labels_numpy = all_val_labels.cpu().numpy()

Epoch 1/30 [Train]:   0%|          | 0/1086 [00:00<?, ?it/s]

Epoch 1/30 [Valid]:   0%|          | 0/272 [00:00<?, ?it/s]

Epoch 1: Train Loss = 0.0190, Val Loss = 0.2164
            Train Acc = 87.5256, Val Acc = 31.0297


Epoch 2/30 [Train]:   0%|          | 0/1086 [00:00<?, ?it/s]

Epoch 2/30 [Valid]:   0%|          | 0/272 [00:00<?, ?it/s]

...

Epoch 14/30 [Train]:   0%|          | 0/1086 [00:00<?, ?it/s]

Epoch 14/30 [Valid]:   0%|          | 0/272 [00:00<?, ?it/s]

Epoch 14: Train Loss = 0.0151, Val Loss = 0.2439
            Train Acc = 90.0366, Val Acc = 28.5879


Epoch 15/30 [Train]:   0%|          | 0/1086 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
torch.save(model, 'EmotionClassifier.pt')

In [ ]:
model1 = torch.load('EmotionClassifier.pt', weights_only=False)